In [1]:
pip install hmmlearn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.0/166.0 kB 2.5 MB/s eta 0:00:00


In [2]:
import yfinance as  yf
import pandas as pd
import numpy as np
import cvxpy as cp
from scipy.stats import norm
import matplotlib.pyplot as plt
from hmmlearn.hmm import GaussianHMM

In [43]:
import statsmodels.api as sm

In [66]:
data = yf.download(defensive, period="2y")["Close"]

/tmp/ipykernel_7832/3704708437.py:1: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(defensive, period="2y")["Close"]
[*********************100%***********************]  2 of 2 completed


In [72]:
mkt_data = yf.download("^GSPC", period="2y")["Close"]

/tmp/ipykernel_7832/3603977230.py:1: FutureWarning: YF.download() has changed argument auto_adjust default to True
  mkt_data = yf.download("^GSPC", period="2y")["Close"]
[*********************100%***********************]  1 of 1 completed


In [77]:
returns = data.pct_change().dropna()
mkt_returns = mkt_data.pct_change().dropna()
idx = returns.index.intersection(mkt_returns.index)
y = returns.loc[idx] - 0.003
x = mkt_returns.loc[idx] - 0.003

/tmp/ipykernel_7832/1063385934.py:1: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  returns = data.pct_change().dropna()


In [96]:
X = sm.add_constant(x)
for i in range(len(returns.columns)):
  model = sm.OLS(y.iloc[:, i], X).fit()
  alpha = model.params['const']
  beta = model.params['^GSPC']
  r_squared = model.rsquared

  # 7. Print the specific outputs
  print(f"Alpha (Intercept): {alpha:.6f}")
  print(f"Beta (Coefficient): {beta:.4f}")
  print(f"R-Squared (R2):     {r_squared:.4f}")

Alpha (Intercept): -0.002852
Beta (Coefficient): -0.0001
R-Squared (R2):     0.0000
Alpha (Intercept): -0.002968
Beta (Coefficient): -0.0004
R-Squared (R2):     0.0000


In [9]:
class Selector:
  def __init__(self, rf:float=0.01, T:int=252, corr_threshold:int=.85):
    self.corr_threshold = corr_threshold
    self.rf = rf/T

  def hmm_score(self, returns, hmm_params, risk_q, recalc_window):
    scores = {}

    hmm = GaussianHMM(**hmm_params)
    hmm.fit(returns)

    transmat = hmm.transmat_
    state_probs = hmm.predict_proba(returns)[-1]

    expected_probs =  state_probs @ (np.linalg.matrix_power(transmat, recalc_window))

    for i, c in enumerate(returns.columns):
      asset_means = hmm.means_[:, i]
      if hmm_params.get("covariance_type", "diag") == "diag":
          asset_vars = hmm.covars_[:, i]
      else:
          raise ValueError("Only 'diag' covariance type is explicitly supported in this vector loop.")

      scores[c] = self.calculate_score(
          asset_means,
          asset_vars,
          expected_probs,
          risk_q,
          recalc_window
      )

    return scores, expected_probs

  def calculate_score(self, state_means, state_vars, expected_probs, risk_q, recalc_window):
    tail_param = norm.pdf(norm.ppf(risk_q))/(1-risk_q)
    ES = -state_means + np.sqrt(state_vars) * tail_param

    blend_es = expected_probs @ ES
    blend_mean = expected_probs @ state_means

    tres = blend_mean / np.maximum(blend_es, 1e-6)

    return tres

  def get_current_state(self, returns, hmm_params, risk_q, recalc_window):
    hmm = GaussianHMM(**hmm_params)
    hmm.fit(returns)

    transmat = hmm.transmat_
    state_probs = hmm.predict_proba(returns)[-1]

  def hmm_filter(self, data, hmm_params, risk_q, recalc_window, selection_q=0.6):
    returns = data.pct_change().dropna()
    scores, expected_probs = self.hmm_train(returns, hmm_params, risk_q, recalc_window)

    score_series = pd.Series(scores)

    threshold = score_series.quantile(selection_q)
    valid_assets = score_series[score_series > threshold].index.tolist()

    return data[valid_assets], expected_probs



  def corr_filter(self, data):
    corr_matrix = data.pct_change().dropna().corr()
    std = data.std()

    sharpe = (np.mean(data.pct_change())-self.rf) / std

    drop = []
    for ticker in data.columns:
      if ticker in drop:
        continue
      ticker_idx = data.columns.get_loc(ticker)

      for i in data.columns:
        if ticker == i or i in drop:
          continue

        i_idx = data.columns.get_loc(i)
        if corr_matrix.iloc[ticker_idx, i_idx] > self.corr_threshold:
          print(corr_matrix.iloc[ticker_idx, i_idx])
          if sharpe[ticker] > sharpe[i]:
            drop.append(i)
          else:
            drop.append(ticker)

    return data.drop(columns=drop)

In [30]:
class Optimizer:
  def __init__(self):
    pass

  def min_var_portfolio(
      self,
      returns,
      blended_mtx,
  ):
    pass

  def mvo_optimize_portfolio(
    self,
    returns,
    sector_exposure_matrix,
    country_exposure_matrix
  ):
    if self.per_sector_penalty is None:
        per_sector_penalty_mtx = np.ones(sector_exposure_matrix.shape[0])
    else:
        per_sector_penalty_mtx = np.array(list(self.per_sector_penalty.values()))

    if self.per_country_penalty is None:
        per_country_penalty_mtx = np.ones(country_exposure_matrix.shape[0])
    else:
        per_country_penalty_mtx = np.array(list(self.per_country_penalty.values()))

    sec_exp_cap = np.array(list(self.max_sector_exposure.values()))
    cnt_exp_cap = np.array(list(self.max_country_exposure.values()))

    mu = np.mean(returns, axis=0)
    S, N = returns.shape
    R = returns.values

    w = cp.Variable(N)
    u = cp.Variable(S)
    zeta = cp.Variable()

    ex_R = mu.values @ w

    es = zeta + (1 / ((1 - self.prct) * S)) * cp.sum(u)

    constraints = [
        u >= -R @ w - zeta,
        u >= 0,
        cp.sum(w) == 1,
        w >= 0,
        sector_exposure_matrix @ w <= sec_exp_cap,
        country_exposure_matrix @ w <= cnt_exp_cap,
    ]

    sector_Q = np.diag(per_sector_penalty_mtx)
    sector_P = sector_exposure_matrix.T @ sector_Q @ sector_exposure_matrix

    sector_penalty_val = cp.quad_form(w, sector_P)

    country_Q = np.diag(per_country_penalty_mtx)
    country_P = country_exposure_matrix.T @ country_Q @ country_exposure_matrix
    country_penalty_val = cp.quad_form(w, country_P)

    total_penalties = (
        self.sector_penalty_weight * sector_penalty_val
        + self.country_penalty_weight * country_penalty_val
    )

    obj_fn = cp.Maximize(ex_R - self.risk_aversion * es - total_penalties)

    problem = cp.Problem(obj_fn, constraints)
    problem.solve(solver=cp.CLARABEL)


    if problem.status not in ["optimal", "feasible"]:
        raise ValueError(f"Optimization failed: {problem.status}")

    weights = np.array(w.value)
    weights[weights < self.min_weight_threshold] = 0.0

    total_w = np.sum(weights)
    if total_w > 0:
      weights_recalc = weights / total_w

    return weights_recalc

In [31]:
class MVOPortfolio(Optimizer):
  def __init__(self, hmm_params):
    super().__init__()
    self.sector_penalty_weight=0.1
    self.country_penalty_weight=0.05
    self.prct=0.95
    self.risk_aversion=3.0
    self.min_weight_threshold=0.01

    self.max_sector_exposure = {
      "Tech": 0.30,
      "Finance": 0.20,
      "Industry": 0.20,
      "Telecom": 0.10,
      "ConsumerGoods": 0.30,
      "Health": 0.2
    }

    self.max_country_exposure = {
      "US": 0.7,
      "Europe": 0.25,
      "Asia": 0.25
    }

    self.per_sector_penalty = {
      "Tech": 1.3,
      "Finance": 1.8,
      "Industry": 1.0,
      "Telecom": 0.8,
      "ConsumerGoods": 0.8,
      "Health": 0.3
    }

    self.per_country_penalty = {
      "US": 1.5,
      "Europe": 1.0,
      "Asia": 1.0
    }

  def download(self, indices, start, end, interval):
    data = {}
    for ticker in indices:
        clean_ticker = ticker.strip()

        try:
            tmp_data = yf.Ticker(clean_ticker).history(start=start, end=end, interval=interval)["Close"]

            if tmp_data.empty:
                print(f"Warning: No data returned for {clean_ticker}")
                continue

            tmp_data.index = pd.to_datetime(tmp_data.index).tz_localize(None).normalize()

            tmp_data = tmp_data[~tmp_data.index.duplicated(keep='last')]

            data[clean_ticker] = tmp_data

        except Exception as e:
            print(f"Failed to download {clean_ticker}: {e}")

    if not data:
        raise ValueError("No data was successfully downloaded for any of the provided tickers.")

    df = pd.concat(data, axis=1)

    return df.sort_index()

  def get_data(
    self,
    indices,
    start,
    end,
    interval="1d",
    benchmark = "^GSPC"
  ):
    returns_path = f"returns_{start}_{end}.parquet"
    benchmark_path = f"benchmark_{start}_{end}.parquet"

    import os
    if not os.path.exists(returns_path) or not os.path.exists(benchmark_path):
      df = self.download(indices, start, end, interval)
      sp500 = self.download([benchmark], start, end, interval)

      self.returns = df.pct_change().dropna()
      self.benchmark = sp500.pct_change().dropna()
      self.returns.to_parquet(returns_path)
      self.benchmark.to_parquet(benchmark_path)

    else:
      self.returns = pd.read_parquet(returns_path)
      self.benchmark = pd.read_parquet(benchmark_path)

  def plot_data(self):
    df_pats = np.cumsum(self.returns*100, axis=0) + 100
    df_pats.plot(figsize=(15, 10))
    plt.show()

  def plot_benchmark(self):
    df_pats = np.cumsum(self.benchmark*100, axis=0) + 100
    df_pats.plot(figsize=(15, 10))
    plt.show()

  def get_exposure(self, exposures, country_alloc):
    exposure_df = pd.DataFrame.from_dict(exposures, orient='index')
    exposure_df.columns = ["Tech", "Finance", "Industry", "Telecom", "ConsumerGoods", "Health"]

    country_df = pd.DataFrame.from_dict(country_alloc, orient='index')
    country_df.columns = ["US", "Europe", "Asia"]

    return exposure_df, country_df

  def calculate_metrics(self, returns_series, trading_days=252):
    cum_returns = (1 + returns_series).cumprod()
    final_return = cum_returns.iloc[-1] - 1

    ann_return = returns_series.mean() * trading_days
    ann_vol = returns_series.std() * np.sqrt(trading_days)
    sharpe = ann_return / ann_vol if ann_vol != 0 else 0

    running_max = cum_returns.cummax()
    drawdowns = (cum_returns - running_max) / running_max
    max_dd = drawdowns.min()

    return {
        "Final Return": final_return,
        "Sharpe Ratio": sharpe,
        "Max Drawdown": max_dd,
    }

  def apply_lot_size_rounding(self, weights, asset_prices, total_portfolio_value=1000):
    prices = np.array(asset_prices)

    ideal_cash_alloc = weights * total_portfolio_value

    ideal_shares = ideal_cash_alloc / prices

    actual_shares = np.round(ideal_shares)

    actual_cash_alloc = actual_shares * prices
    actual_total_spent = np.sum(actual_cash_alloc)

    if actual_total_spent > 0:
        rounded_weights = actual_cash_alloc / actual_total_spent
    else:
        rounded_weights = weights

    return rounded_weights, actual_shares

  def rolling_backtest(
      self,
      sector_exp,
      country_exp,
      train_window=252,
      recalc_window=21,
  ):
      total_periods = len(self.returns)
      oos_portfolio_returns = []
      oos_benchmark_returns = []
      oos_dates = []

      sector_matrix, country_matrix = self.get_exposure(sector_exp, country_exp)

      for start_idx in range(0, total_periods - train_window, recalc_window):
          train_end_idx = start_idx + train_window
          test_end_idx = min(train_end_idx + recalc_window, total_periods)

          train_returns_raw = self.returns.iloc[start_idx:train_end_idx]
          train_returns = Selector().corr_filter(train_returns_raw)

          test_returns = self.returns.loc[:, train_returns.columns].iloc[train_end_idx:test_end_idx]

          if test_returns.empty:
              break

          test_bench = self.benchmark.reindex(test_returns.index).fillna(0)

          sector_mtx_vals = sector_matrix.loc[train_returns.columns].T.values
          country_mtx_vals = country_matrix.loc[train_returns.columns].T.values

          optimal_w = self.optimize_portfolio(
              returns=train_returns,
              sector_exposure_matrix=sector_mtx_vals,
              country_exposure_matrix=country_mtx_vals
          )

          optimal_w_scaled, _ = self.apply_lot_size_rounding(optimal_w, test_returns.iloc[-1])

          period_oos_returns = test_returns.values @ optimal_w_scaled

          oos_portfolio_returns.extend(period_oos_returns)
          oos_benchmark_returns.extend(test_bench.iloc[:, 0].values)
          oos_dates.extend(test_returns.index)

      strat_perf = pd.Series(oos_portfolio_returns, index=oos_dates)
      bench_perf = pd.Series(oos_benchmark_returns, index=oos_dates)

      strat_metrics = self.calculate_metrics(strat_perf)
      bench_metrics = self.calculate_metrics(bench_perf)

      summary_df = pd.DataFrame(
          {
              "Strategy": [
                  f"{strat_metrics['Final Return']*100:.2f}%",
                  f"{strat_metrics['Sharpe Ratio']:.2f}",
                  f"{strat_metrics['Max Drawdown']*100:.2f}%",
              ],
              "S&P 500 Benchmark": [
                  f"{bench_metrics['Final Return']*100:.2f}%",
                  f"{bench_metrics['Sharpe Ratio']:.2f}",
                  f"{bench_metrics['Max Drawdown']*100:.2f}%",
              ],
          },
          index=["Final Return", "Risk-Adjusted Return (Sharpe)", "Worst Drawdown"],
      )

      self.display_results(summary_df, strat_perf, bench_perf)

      return optimal_w, train_returns.columns,

  def display_results(self, summary_table, strat_series, bench_series):
      print(summary_table)
      plt.plot(np.cumsum(strat_series)*100+100, label="strat")
      plt.plot(np.cumsum(bench_series)*100+100, label="bench")
      plt.legend()

In [ ]:
# add temperature
# fix hmm selector

In [32]:
hmm_params = {
    "n_components": 3,
    "n_iter": 50,
    "tol": 1e-2,
    "init_params": "stmc",
    "cov_type": "diag"
}

In [33]:
portfolio = MVOPortfolio(hmm_params)

In [34]:
portfolio.get_data(indices, start, end, interval, hmm_params)